In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

if not os.getenv("GOOGLE_API_KEY"):
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

print("Setup complete.")

Setup complete.


In [2]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

# Shared corpus from Notebook 1 — extended for richer demo
corpus = [
    "Transformers use self-attention mechanisms to process sequences in parallel.",
    "BERT is a bidirectional encoder trained using masked language modelling.",
    "The BM25 algorithm ranks documents based on term frequency and inverse document frequency.",
    "Gradient descent is an optimization technique used to minimize the loss function.",
    "Neural networks learn by adjusting weights through backpropagation.",
    "Retrieval Augmented Generation combines a retriever with a language model to produce grounded answers.",
    "Fine-tuning adapts a pre-trained model to a specific downstream task using task-specific data.",
    "Quantization reduces model size by representing weights in lower bit formats such as INT8.",
    "The attention mechanism computes a weighted sum of value vectors based on query-key similarity.",
    "Large language models are trained on massive text corpora to learn general-purpose representations.",
]

print(f"Corpus: {len(corpus)} documents loaded.")

C:\Users\Meghana Veeramallu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Corpus: 10 documents loaded.


In [3]:
# Load cross-encoder model (much smaller than full BERT, fast enough for top-K re-ranking)
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Load bi-encoder for first-stage retrieval
bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Models loaded.")
print("  Bi-encoder:    all-MiniLM-L6-v2 (first-stage retrieval)")
print("  Cross-encoder: ms-marco-MiniLM-L-6-v2 (re-ranking)")

C:\Users\Meghana Veeramallu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Meghana Veeramallu\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings

Models loaded.
  Bi-encoder:    all-MiniLM-L6-v2 (first-stage retrieval)
  Cross-encoder: ms-marco-MiniLM-L-6-v2 (re-ranking)


In [4]:
# ---- Stage 1: Bi-Encoder Retrieval ----

query = "How does attention work in language models?"

doc_vecs   = bi_encoder.encode(corpus, convert_to_numpy=True)
doc_vecs   = doc_vecs / np.linalg.norm(doc_vecs, axis=1, keepdims=True)
query_vec  = bi_encoder.encode([query], convert_to_numpy=True)[0]
query_vec  = query_vec / np.linalg.norm(query_vec)

bi_scores  = doc_vecs @ query_vec
bi_ranked  = np.argsort(bi_scores)[::-1][:5]  # Top-5 from bi-encoder

print("STAGE 1 — Bi-Encoder Top-5 Candidates")
print("-" * 70)
for rank, idx in enumerate(bi_ranked, 1):
    print(f"  #{rank}  [score={bi_scores[idx]:.4f}] {corpus[idx]}")

STAGE 1 — Bi-Encoder Top-5 Candidates
----------------------------------------------------------------------
  #1  [score=0.5273] The attention mechanism computes a weighted sum of value vectors based on query-key similarity.
  #2  [score=0.4970] Large language models are trained on massive text corpora to learn general-purpose representations.
  #3  [score=0.3708] BERT is a bidirectional encoder trained using masked language modelling.
  #4  [score=0.3478] Transformers use self-attention mechanisms to process sequences in parallel.
  #5  [score=0.3418] Fine-tuning adapts a pre-trained model to a specific downstream task using task-specific data.


In [5]:
# ---- Stage 2: Cross-Encoder Re-Ranking ----

# Build (query, doc) pairs for the cross-encoder
candidate_docs = [corpus[idx] for idx in bi_ranked]
query_doc_pairs = [[query, doc] for doc in candidate_docs]

# Cross-encoder scores (logit; higher = more relevant)
ce_scores = cross_encoder.predict(query_doc_pairs)

# Re-rank candidates
reranked_order = np.argsort(ce_scores)[::-1]

print("STAGE 2 — Cross-Encoder Re-Ranking")
print("-" * 70)
for rank, idx_in_candidates in enumerate(reranked_order, 1):
    original_doc_id = bi_ranked[idx_in_candidates]
    print(f"  #{rank}  [CE score={ce_scores[idx_in_candidates]:.4f}] {candidate_docs[idx_in_candidates]}")

print("\n--- Before vs After Re-Ranking ---")
print(f"{'Bi-Encoder Rank 1:':<30} {candidate_docs[0]}")
print(f"{'Cross-Encoder Rank 1:':<30} {candidate_docs[reranked_order[0]]}")

STAGE 2 — Cross-Encoder Re-Ranking
----------------------------------------------------------------------
  #1  [CE score=3.3522] The attention mechanism computes a weighted sum of value vectors based on query-key similarity.
  #2  [CE score=-4.4683] Large language models are trained on massive text corpora to learn general-purpose representations.
  #3  [CE score=-5.4407] Transformers use self-attention mechanisms to process sequences in parallel.
  #4  [CE score=-6.9237] BERT is a bidirectional encoder trained using masked language modelling.
  #5  [CE score=-10.4939] Fine-tuning adapts a pre-trained model to a specific downstream task using task-specific data.

--- Before vs After Re-Ranking ---
Bi-Encoder Rank 1:             The attention mechanism computes a weighted sum of value vectors based on query-key similarity.
Cross-Encoder Rank 1:          The attention mechanism computes a weighted sum of value vectors based on query-key similarity.


In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)

# HyDE prompt: generate a hypothetical document that would answer this query
hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical writer. Generate a single factual paragraph (3-5 sentences) that would directly answer the following question. Write it as if it were an excerpt from a textbook."),
    ("human", "{query}")
])

hyde_chain = hyde_prompt | llm | StrOutputParser()

query = "How does attention work in language models?"

# Without HyDE: embed the raw query
raw_query_vec = bi_encoder.encode([query], convert_to_numpy=True)[0]

# With HyDE: generate hypothetical doc, then embed it
hypothetical_doc = hyde_chain.invoke({"query": query})

print("Original Query:")
print(f"  '{query}'")
print(f"\nHypothetical Document (generated by Gemini):")
print(f"  {hypothetical_doc}")

Original Query:
  'How does attention work in language models?'

Hypothetical Document (generated by Gemini):
  Attention mechanisms in language models allow the model to dynamically weigh the importance of different parts of an input sequence when processing each element. This process involves computing a "query" vector for the current token, which is then compared against "key" vectors for all other tokens in the sequence to determine their relevance. The resulting similarity scores are normalized, typically via a softmax function, and then used as weights to create a weighted sum of "value" vectors associated with each token. This weighted sum forms a new, context-aware representation for the current token, effectively enabling the model to focus on the most pertinent information from the entire input to inform its understanding and generation. This mechanism is crucial for capturing long-range dependencies and intricate contextual relationships within text.


In [9]:
# Compare retrieval: raw query vs HyDE
hyde_vec = bi_encoder.encode([hypothetical_doc], convert_to_numpy=True)[0]
hyde_vec = hyde_vec / np.linalg.norm(hyde_vec)

raw_vec  = raw_query_vec / np.linalg.norm(raw_query_vec)

raw_scores  = doc_vecs @ raw_vec
hyde_scores = doc_vecs @ hyde_vec

raw_ranked  = np.argsort(raw_scores)[::-1][:3]
hyde_ranked = np.argsort(hyde_scores)[::-1][:3]

print(f"Query: '{query}'\n")
print("Raw Query Retrieval (Top 3):")
for r, idx in enumerate(raw_ranked, 1):
    print(f"  #{r} [score={raw_scores[idx]:.4f}] {corpus[idx]}")

print("\nHyDE Retrieval (Top 3):")
for r, idx in enumerate(hyde_ranked, 1):
    print(f"  #{r} [score={hyde_scores[idx]:.4f}] {corpus[idx]}")

Query: 'How does attention work in language models?'

Raw Query Retrieval (Top 3):
  #1 [score=0.5273] The attention mechanism computes a weighted sum of value vectors based on query-key similarity.
  #2 [score=0.4970] Large language models are trained on massive text corpora to learn general-purpose representations.
  #3 [score=0.3708] BERT is a bidirectional encoder trained using masked language modelling.

HyDE Retrieval (Top 3):
  #1 [score=0.6712] The attention mechanism computes a weighted sum of value vectors based on query-key similarity.
  #2 [score=0.5350] Large language models are trained on massive text corpora to learn general-purpose representations.
  #3 [score=0.4125] Retrieval Augmented Generation combines a retriever with a language model to produce grounded answers.


In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Step 1: Use Gemini to generate 3 alternative phrasings of the query
multi_query_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an AI assistant helping improve document retrieval.
Given a user question, generate 3 different paraphrases of that question.
Each paraphrase should approach the topic from a slightly different angle.
Return ONLY the 3 questions, one per line. No numbering, no extra text."""),
    ("human", "{query}")
])

multi_query_chain = multi_query_prompt | llm | StrOutputParser()

query = "How does attention work in language models?"

# Generate query variants
variants_raw = multi_query_chain.invoke({"query": query})
query_variants = [q.strip() for q in variants_raw.strip().split("\n") if q.strip()]
all_queries = [query] + query_variants[:3]  # original + up to 3 variants

print("Generated query variants:")
for i, q in enumerate(all_queries):
    label = "(original)" if i == 0 else f"(variant {i})"
    print(f"  {label}: {q}")

# Step 2: Retrieve top-3 for each query variant, then deduplicate
seen = set()
all_retrieved = []

for q in all_queries:
    q_vec   = bi_encoder.encode([q], convert_to_numpy=True)[0]
    q_vec   = q_vec / np.linalg.norm(q_vec)
    scores  = doc_vecs @ q_vec
    top_idx = np.argsort(scores)[::-1][:3]
    for idx in top_idx:
        text = corpus[idx]
        if text not in seen:
            seen.add(text)
            all_retrieved.append({"doc_id": int(idx), "text": text, "from_query": q})

print(f"\nMulti-Query retrieved {len(all_retrieved)} unique documents (union of all variants):")
for doc in all_retrieved:
    print(f"  [doc_{doc['doc_id']}] {doc['text']}")

Generated query variants:
  (original): How does attention work in language models?
  (variant 1): Describe the mechanism of attention within language models.
  (variant 2): What role does attention play in the operation of language models?
  (variant 3): How does attention enhance the capabilities of language models?

Multi-Query retrieved 5 unique documents (union of all variants):
  [doc_8] The attention mechanism computes a weighted sum of value vectors based on query-key similarity.
  [doc_9] Large language models are trained on massive text corpora to learn general-purpose representations.
  [doc_1] BERT is a bidirectional encoder trained using masked language modelling.
  [doc_0] Transformers use self-attention mechanisms to process sequences in parallel.
  [doc_6] Fine-tuning adapts a pre-trained model to a specific downstream task using task-specific data.


In [11]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# --- Reuse HybridRetriever from Notebook 1 ---
class HybridRetriever:
    def __init__(self, corpus, k=60):
        self.corpus = corpus
        self.k = k
        tokenized = [doc.lower().split() for doc in corpus]
        self.bm25  = BM25Okapi(tokenized)
        sbert      = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        doc_vecs   = sbert.encode(corpus, convert_to_numpy=True)
        self.doc_vecs = doc_vecs / np.linalg.norm(doc_vecs, axis=1, keepdims=True)
        self.sbert = sbert

    def retrieve(self, query, top_k=5):
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranked = np.argsort(bm25_scores)[::-1]
        bm25_ranks  = {int(d): r+1 for r, d in enumerate(bm25_ranked)}

        q_vec       = self.sbert.encode([query], convert_to_numpy=True)[0]
        q_vec       = q_vec / np.linalg.norm(q_vec)
        sbert_scores = self.doc_vecs @ q_vec
        sbert_ranked = np.argsort(sbert_scores)[::-1]
        sbert_ranks  = {int(d): r+1 for r, d in enumerate(sbert_ranked)}

        rrf = {d: 1/(self.k+bm25_ranks[d]) + 1/(self.k+sbert_ranks[d]) for d in range(len(self.corpus))}
        final = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
        return [self.corpus[d] for d in final]

hybrid = HybridRetriever(corpus)

# --- Step functions ---
def expand_query(query: str) -> str:
    """Use HyDE to expand the query."""
    return hyde_chain.invoke({"query": query})

def hybrid_retrieve(expanded_query: str) -> list[str]:
    """Retrieve top-5 using Hybrid (BM25+SBERT+RRF)."""
    return hybrid.retrieve(expanded_query, top_k=5)

def rerank(data: dict) -> list[str]:
    """Re-rank retrieved docs using cross-encoder."""
    query = data["original_query"]
    docs  = data["candidates"]
    pairs  = [[query, doc] for doc in docs]
    scores = cross_encoder.predict(pairs)
    ranked = np.argsort(scores)[::-1][:3]  # Keep top-3
    return [docs[i] for i in ranked]

def format_context(docs: list[str]) -> str:
    return "\n\n".join(f"[Document {i+1}]\n{doc}" for i, doc in enumerate(docs))

print("Pipeline components defined.")

Pipeline components defined.


In [12]:
# --- Final Answer Generation Prompt ---
generation_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a knowledgeable AI assistant. Answer the user's question using ONLY the provided context documents.
If the answer is not in the context, say 'I don't have enough information to answer this.'
Be concise and precise.

Context:
{context}"""),
    ("human", "{question}")
])

# --- Full pipeline using LCEL ---
def run_advanced_rag(user_query: str):
    print(f"Query: '{user_query}'")
    print("=" * 70)

    # Step 1: Query Expansion (HyDE)
    expanded = expand_query(user_query)
    print(f"[1] HyDE Expansion: {expanded[:150]}...")

    # Step 2: Hybrid Retrieval on expanded query
    candidates = hybrid_retrieve(expanded)
    print(f"\n[2] Hybrid Retrieval ({len(candidates)} candidates):")
    for i, c in enumerate(candidates, 1):
        print(f"     #{i}: {c[:80]}")

    # Step 3: Cross-Encoder Re-Ranking
    top_docs = rerank({"original_query": user_query, "candidates": candidates})
    print(f"\n[3] After Re-Ranking (top 3):")
    for i, d in enumerate(top_docs, 1):
        print(f"     #{i}: {d}")

    # Step 4: LLM Generation
    context = format_context(top_docs)
    chain   = generation_prompt | llm | StrOutputParser()
    answer  = chain.invoke({"context": context, "question": user_query})

    print(f"\n[4] Final Answer:")
    print(f"     {answer}")
    return answer

# --- Run it ---
run_advanced_rag("How does attention work in language models?")

Query: 'How does attention work in language models?'
[1] HyDE Expansion: Attention mechanisms in language models enable the model to dynamically weigh the importance of different parts of an input sequence when processing a...

[2] Hybrid Retrieval (5 candidates):
     #1: The attention mechanism computes a weighted sum of value vectors based on query-
     #2: Large language models are trained on massive text corpora to learn general-purpo
     #3: Fine-tuning adapts a pre-trained model to a specific downstream task using task-
     #4: Retrieval Augmented Generation combines a retriever with a language model to pro
     #5: Transformers use self-attention mechanisms to process sequences in parallel.

[3] After Re-Ranking (top 3):
     #1: The attention mechanism computes a weighted sum of value vectors based on query-key similarity.
     #2: Large language models are trained on massive text corpora to learn general-purpose representations.
     #3: Transformers use self-attention mec

"I don't have enough information to answer this."

In [13]:
# Try a second query
run_advanced_rag("What is the purpose of backpropagation?")

Query: 'What is the purpose of backpropagation?'
[1] HyDE Expansion: Backpropagation is a foundational algorithm primarily used to train artificial neural networks by efficiently calculating the gradient of the loss fun...

[2] Hybrid Retrieval (5 candidates):
     #1: Gradient descent is an optimization technique used to minimize the loss function
     #2: Neural networks learn by adjusting weights through backpropagation.
     #3: The attention mechanism computes a weighted sum of value vectors based on query-
     #4: The BM25 algorithm ranks documents based on term frequency and inverse document 
     #5: BERT is a bidirectional encoder trained using masked language modelling.

[3] After Re-Ranking (top 3):
     #1: Neural networks learn by adjusting weights through backpropagation.
     #2: Gradient descent is an optimization technique used to minimize the loss function.
     #3: The attention mechanism computes a weighted sum of value vectors based on query-key similarity.

[4] F

'Neural networks learn by adjusting weights through backpropagation.'